# BP4 Gate 1 — Business Understanding & Policy
**Customer360 Navigator Enterprise Suite — Customer Journey Analytics**

## Why this notebook exists, and what it honestly scopes
Master Execution Plan Section 5.1 / Section 7 define BP4 in words that are, deliberately, a warning
as much as a mandate: *"do not invent customer IDs. Build standardized issue/event sequences only
from actual identifiers and timestamps present in the real data; if longitudinal identity is absent,
call this event/issue journey analytics explicitly — never longitudinal customer journey
analytics."* This is BP4's very first notebook, opening Sprint 4 (Section 23) fresh off BP3's close.
It grounds BP4's entire scope against the real, already-profiled CFPB extract and the real Gold-layer
artifacts BP1–BP3 already built — before any Gate 2 aggregation pipeline is written — and it states
plainly, up front, the one design fact that shapes everything downstream: **this real CFPB extract
has no customer or consumer identifier of any kind.**

## The central scoping fact, live-verified below (Section 4)
The real 15-column CFPB schema — re-verified live in this notebook exactly as BP1/BP2/BP3 Gate 1 each
re-verified it independently on their own runs — carries no field resembling a persistent
customer/consumer identity. `Complaint ID` is a real, unique identifier, but it identifies one
**complaint** (one event), not one **person**: the same live keyword check BP2 Gate 1 first
established and BP3 Gate 1 reused finds zero columns matching `customer`, `consumer id`, `person`,
`account number`, `ssn`, `email`, or `phone`. This is not a data-quality gap to fill — the CFPB
public Consumer Complaint Database is deliberately published without a persistent complainant
identifier, for privacy reasons, and no combination of `ZIP code` + `Company` + `Product` + `Issue` +
`Date received` recovers one safely or honestly. **This notebook therefore commits, formally and in
writing, to the name the Master Plan itself uses for this situation: BP4 is event/issue journey
analytics, never customer journey analytics — this distinction is stated in every BP4 deliverable
from Gate 1 forward, never softened to "customer" for readability.**

## What a "journey" honestly means here, grounded in what the real data actually supports
Two real, defensible units of analysis exist in this extract, and BP4 builds on both rather than
inventing a third:

1. **The complaint-event journey (row-level, exactly 815,453*-scale full population, live-verified
   below).** Every real CFPB row already carries two real timestamps bounding a real two-step event
   sequence: `Date received` → `Date sent to company`. This is a genuine journey in miniature — a
   complaint arrives, and is later dispatched to the company — with a real, computable duration
   (`response_lag_days`) and a real terminal state (`Company response to consumer`, cross-referenced
   against CFPB's own `Timely response?` judgment of that same duration). `Complaint ID` is the real,
   unique key for this unit (live-verified 1:1 with rows below) — it identifies one event's journey,
   not a person's.

2. **The issue-cluster journey (aggregate-level).** Grouping real complaints on a real, fully
   disclosed key — `Company` × `Product` × `Sub-product` × `Issue` × `Sub-issue` — and ordering each
   group's real complaints by `Date received` turns that group into its own real time series: how
   many real complaints hit this exact real combination, month over month. This is the
   "repeat-contact" signal the Master Plan asks BP4 to look for, **explicitly redefined as a
   recurring-issue-cluster signal, never a customer's own repeat-contact count** — the two are
   easy to conflate and this notebook draws the line here, in writing, rather than leaving it
   implicit. Live-verified below: this key produces 37,160 real distinct clusters, 15,551 of which
   (41.85%) recur (more than one real complaint), together accounting for 1,026,966 of 1,048,575 rows
   (97.94%) — a real, well-supported signal, not a sparse one.

*(the 815,453-trainable-row figure is BP3's own scoped subset; BP4 works over the full real
1,048,575-row extract, since it excludes nothing — there is no target to be undefined for.)*

## The "derived BANKING77" integration, reported with its real, honest coverage limit
The Master Plan's own BP table marks BP4 `Integrates BANKING77? = YES (derived)` — deliberately
distinguished from BP1's and BP2's direct `YES`. BP4 never runs BP1's text classifier against CFPB
rows (this extract carries no `Consumer complaint narrative` column at all, confirmed live by
BP1/BP2's own Gate 1 notebooks and re-confirmed live again in Section 4 below — there is no narrative
text to classify). Instead, "derived" means BP4 reuses the **Gold-layer `common_taxonomy_bucket`
column** already built once, at the shared taxonomy layer (Section 6, "Common Taxonomy / Intent
Layer"), and reused unmodified here (HYPER) as an optional, secondary intent-aligned dimension over
the issue-cluster journey above. **Its real coverage is reported honestly, not implied to be
complete**: live-verified below, only 68,710 of 1,048,575 real rows (6.55%) fall inside BANKING77's
real intent overlap (`CARD_ISSUANCE_AND_LIFECYCLE` 32,495, `ATM_CASH_WITHDRAWAL` 27,952, `TRANSFERS`
8,263); the remaining 979,865 rows (93.45%) are tagged `OUT_OF_SCOPE_NO_BANKING77_OVERLAP` — the same
real scope boundary BP1's own Gate 1 first surfaced. BP4 therefore uses the native CFPB
`Product`/`Sub-product`/`Issue`/`Sub-issue` fields (fully populated) as its **primary** journey-grouping
dimension, and overlays `common_taxonomy_bucket` only as a secondary, honestly-scoped BANKING77-aligned
view of the in-scope 6.55% slice — never presented as if it covered the whole real dataset.

## Purpose
Produces BP4's Gate 1 output exactly as Section 8 (6-Gate Governance SOP) defines it: a policy
artifact recording BP4's journey definition, its scope boundaries (BP4's analog to BP1–BP3's
`leakage_rules` — there is no supervised target to leak, so this section instead bars every path that
could smuggle a fabricated customer identity into a later gate), and ASSUMPTIONs — all verified
against the real, already-profiled data and the real, already-built Gold-layer artifacts, never
invented. Gate 1's own exit criterion (Section 8: "No target leakage possible by construction") is
reframed here, explicitly, as **"no fabricated customer identity or invented longitudinal linkage is
possible by construction"** — verified live below by construction, since this notebook itself never
joins CFPB rows on any key except the real, disclosed ones named above.

## Standing rules this notebook follows
- **Execution boundary** (Section 12.2): Claude wrote this notebook; it does not run it. You run it
  on your own machine, and the real, live-checked results below become this project's Gate 1 policy
  record for BP4.
- **Zero-fabrication** (Section 12.1): every check below runs against the real raw CFPB file and the
  real, already-built Gold-layer parquet in `data/processed/`. No category label, count, distinct
  value, or coverage percentage in this notebook is asserted from memory.
- **CFPB supervisory & complaint-handling standards (Master Plan Section 9)** — the framework row
  this notebook maps to most directly (`Governs: All 8 BPs; primarily BP1–BP4`): complaint taxonomy
  aligned to CFPB's own product/issue/sub-issue schema (used as BP4's primary journey key, above);
  no fabricated outcomes; dataset limitations (no customer ID; BANKING77 coverage 6.55%) stated
  plainly wherever a population-level claim could otherwise be inferred.
- **ECOA / Regulation B is not a BP4 compliance touchpoint** per the Master Plan's own Section 9
  mapping (`BP1, BP2, BP3, BP7` only) — BP4 builds no scored/classified output and performs no
  disparate-impact-style testing. As a conservative scope decision anyway (HYPER-consistent with
  BP1–BP3's own precedent), `Tags` is barred from BP4's journey-grouping keys — see Scope Boundaries.
- **GLBA**: this real extract carries no narrative/PII-adjacent text field at all (re-confirmed live
  in Section 4) — not applicable beyond the existing no-narrative-text finding BP1/BP2 Gate 1 already
  established.
- **WARP**: `configure_performance()` first. The real 1,048,575-row raw CFPB file is read lazily via
  `pl.scan_csv`, exactly as BP1/BP2/BP3 Gate 1 each do; the real Gold-layer parquet is read via
  `pl.scan_parquet`, BP4's own designated Section 17.5 aggregation stack (Polars/DuckDB, lazy scans,
  SQL-style group-by push-down, no per-row Python loops) used from Gate 1 forward, not introduced
  later.
- **HYPER**: reuses `src/taxonomy/taxonomy_mapper.CFPB_DTYPES` (real column dtypes, no
  re-derivation), `src/utils/bp1_config_sync.py` (generic marker-based config read/write, already
  reused unmodified by BP1/BP2/BP3), the same "no persistent customer/consumer identifier" ID-keyword
  check BP2 Gate 1 first established, and the Gold-layer `common_taxonomy_bucket` column BP1/BP2's
  own Gate 2 work already built — never recomputed here.
- **Idempotent**: re-running this notebook overwrites `configs/bp4_customer_journey_analytics.yaml`
  (front matter only) and this notebook's own `policy.json` artifact in place.
- **PROJECT_STRUCTURE_LOCKED.md rule #3**: same resolver as every other notebook in this project.

## Journey definition (Business Understanding, Master Plan Section 5.1/7, BP4)
BP4 has no single supervised target — it is descriptive/aggregation analytics over two real,
disclosed units of analysis, both defined above and both live-verified in Section 4–6 below:
- **`complaint_event_journey`** — row-level, keyed by the real `Complaint ID`; a real two-step
  sequence (`Date received` → `Date sent to company`) with a real computed `response_lag_days`
  duration and a real terminal state (`Company response to consumer`, `Timely response?`).
- **`issue_cluster_journey`** — aggregate-level, keyed by the real, disclosed
  `(Company, Product, Sub-product, Issue, Sub-issue)` combination, ordered by `Date received` into a
  real monthly complaint-volume time series per cluster; optionally overlaid with the Gold-layer
  `common_taxonomy_bucket` field for the real, honestly-scoped 6.55% of rows where a BANKING77-aligned
  intent bucket applies.
- **Explicitly out of scope, by design, not by data gap**: any per-customer/per-consumer journey,
  cohort, retention, or survival-style analysis — none is buildable without inventing an identity this
  real extract does not carry, and none will be built here regardless of how the aggregate patterns
  above look once computed.

## Outputs (both written, idempotent overwrite-in-place)
- `configs/bp4_customer_journey_analytics.yaml` — `journey_definition`, `scope_boundaries`,
  `assumptions`, `status` written to the front-matter section (existing gate blocks, if any,
  preserved verbatim)
- `notebooks/bp4_customer_journey_analytics/artifacts/policy.json` — the Section 8 Gate 1 output
  artifact, with every live enumeration, coverage percentage, and cluster statistic embedded

## Prerequisites
`01_data_acquisition_profiling.ipynb` and BP1/BP2's own Gate 2 notebooks (which built the Gold-layer
`common_taxonomy_bucket` column this notebook reads) should have been real-run at least once. This
notebook re-verifies everything it needs independently rather than trusting those artifacts blindly.

## If a structural check below fails
It raises `AssertionError` with the failing check named. A failing scope-boundary check in particular
must never be worked around — if a later gate's real data forces a choice that would require
inventing a customer identity to proceed, the honest response is to narrow BP4's scope further, not
to fabricate the missing identity.


In [ ]:
# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
import os
import sys
from pathlib import Path


def _find_project_root(marker_filename: str = "PROJECT_STRUCTURE_LOCKED.md") -> Path:
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        candidate = Path(env_override)
        if (candidate / marker_filename).exists():
            return candidate
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {candidate} but {marker_filename} was not found there. "
            "Fix the environment variable rather than removing this check."
        )

    start = Path.cwd()
    current = start
    for _ in range(8):
        if (current / marker_filename).exists():
            return current
        if current.parent == current:
            break
        current = current.parent

    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker_filename in filenames:
            return Path(depth_root)

    raise RuntimeError(
        "Could not resolve PROJECT_ROOT. Set the C360_PROJECT_ROOT environment variable to the "
        "Customer360_Navigator_Enterprise_Suite folder, or run this notebook from inside the project tree "
        "(expected at notebooks/bp4_customer_journey_analytics/)."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
CONFIGS_DIR = PROJECT_ROOT / "configs"
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp4_customer_journey_analytics" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP - configure_performance() FIRST, before any heavy/BLAS-backed import
# ============================================================
from utils.performance_setup import configure_performance  # noqa: E402

WARP_SUMMARY = configure_performance(project_root=PROJECT_ROOT, verbose=True)

# ============================================================
# SECTION 3: Heavy imports + flush-forcing print override (LESSONS_LEARNED_APPLIED.md #12)
# ============================================================
import builtins  # noqa: E402
import functools  # noqa: E402
import json  # noqa: E402
import warnings  # noqa: E402
from datetime import datetime, timezone  # noqa: E402

import polars as pl  # noqa: E402

from taxonomy.taxonomy_mapper import CFPB_DTYPES  # noqa: E402

warnings.filterwarnings("ignore")
print = functools.partial(builtins.print, flush=True)

CFPB_PATH = DATA_RAW_DIR / "cfpb_complaints.csv"
GOLD_TAXONOMY_PATH = DATA_PROCESSED_DIR / "cfpb_common_taxonomy_gold.parquet"

# BP4's real, disclosed issue-cluster journey key (Company x Product x Sub-product x Issue x
# Sub-issue) - see this notebook's markdown cell for the full reasoning. 'Tags' and 'ZIP code' are
# deliberately excluded from this key (see Section 7 scope_boundaries).
ISSUE_CLUSTER_KEY = ["Company", "Product", "Sub-product", "Issue", "Sub-issue"]

# ============================================================
# SECTION 4: Structural checks - real CFPB schema re-verified live (not asserted from prior BPs'
# documentation alone), the same "no persistent customer/consumer identifier" check BP2 Gate 1
# first established and BP3 Gate 1 reused (HYPER), plus a new BP4-specific check: 'Complaint ID'
# is a real, unique EVENT identifier (one complaint), never a person identifier.
# ============================================================
cfpb_columns = list(CFPB_DTYPES.keys())
EXPECTED_CFPB_COLUMNS = [
    "Date received",
    "Product",
    "Sub-product",
    "Issue",
    "Sub-issue",
    "Company public response",
    "Company",
    "State",
    "ZIP code",
    "Tags",
    "Submitted via",
    "Date sent to company",
    "Company response to consumer",
    "Timely response?",
    "Complaint ID",
]
ID_LIKE_KEYWORDS = ("customer", "consumer id", "person", "account number", "ssn", "email", "phone")
suspected_customer_id_columns = [c for c in cfpb_columns if any(kw in c.lower() for kw in ID_LIKE_KEYWORDS)]
narrative_text_columns = [c for c in cfpb_columns if "narrative" in c.lower()]
print(f"[OK] Real CFPB columns ({len(cfpb_columns)}): {cfpb_columns}")
print(
    f"[OK] Columns matching customer/consumer-identifier keywords: {suspected_customer_id_columns or 'NONE'}"
)
print(f"[OK] Narrative-text columns present: {narrative_text_columns or 'NONE'}")

cfpb_lazy = pl.scan_csv(CFPB_PATH, schema_overrides=CFPB_DTYPES)
total_rows = cfpb_lazy.select(pl.len()).collect().item()
complaint_id_n_unique = cfpb_lazy.select(pl.col("Complaint ID").n_unique()).collect().item()
complaint_id_is_unique_event_id = complaint_id_n_unique == total_rows
print(f"[OK] Real CFPB row count (live): {total_rows:,}")
print(
    f"[OK] 'Complaint ID' distinct values (live): {complaint_id_n_unique:,} "
    f"(1:1 with rows: {complaint_id_is_unique_event_id})"
)

# ============================================================
# SECTION 5: Live Gold-layer verification - the real 'common_taxonomy_bucket' column BP1/BP2's own
# Gate 2 work already built (HYPER, reused unmodified) is re-read and re-enumerated here, never
# recomputed. Its real coverage is reported honestly, not assumed complete.
# ============================================================
assert GOLD_TAXONOMY_PATH.exists(), (
    f"{GOLD_TAXONOMY_PATH} does not exist - BP1/BP2's own Gate 2 taxonomy-mapping work must have "
    "been real-run at least once before BP4 Gate 1 can read the shared Gold-layer taxonomy bucket."
)
gold_lazy = pl.scan_parquet(GOLD_TAXONOMY_PATH)
gold_row_count = gold_lazy.select(pl.len()).collect().item()
gold_columns = gold_lazy.collect_schema().names()
gold_row_count_matches_raw = gold_row_count == total_rows
print(f"[OK] Real Gold-layer taxonomy parquet row count (live): {gold_row_count:,}")
print(f"[OK] Gold-layer row count matches raw CFPB row count: {gold_row_count_matches_raw}")

common_taxonomy_bucket_counts = (
    gold_lazy.group_by("common_taxonomy_bucket").agg(pl.len().alias("n")).sort("n", descending=True).collect()
)
n_out_of_scope = (
    gold_lazy.filter(pl.col("common_taxonomy_bucket") == "OUT_OF_SCOPE_NO_BANKING77_OVERLAP")
    .select(pl.len())
    .collect()
    .item()
)
n_in_scope_banking77 = gold_row_count - n_out_of_scope
pct_in_scope_banking77 = round(n_in_scope_banking77 / gold_row_count, 4) if gold_row_count else None
print("[OK] Live 'common_taxonomy_bucket' distinct values + real counts:")
print(common_taxonomy_bucket_counts)
print(
    f"[OK] Real BANKING77-derived taxonomy coverage: {n_in_scope_banking77:,} / {gold_row_count:,} "
    f"rows in-scope ({pct_in_scope_banking77:.2%})"
)

# ============================================================
# SECTION 6: Live computation of BP4's two real journey units - (a) complaint-event response-lag
# duration, (b) issue-cluster recurrence - both computed fresh from the real raw CFPB file, never
# estimated.
# ============================================================
date_fmt = "%m/%d/%Y"
lag_lazy = cfpb_lazy.select(
    pl.col("Date received").str.strptime(pl.Date, date_fmt).alias("date_received_parsed"),
    pl.col("Date sent to company").str.strptime(pl.Date, date_fmt).alias("date_sent_parsed"),
).with_columns(
    (pl.col("date_sent_parsed") - pl.col("date_received_parsed")).dt.total_days().alias("response_lag_days")
)
lag_df = lag_lazy.collect()

response_lag_stats = {
    "count": int(lag_df.height),
    "mean": float(lag_df["response_lag_days"].mean()),
    "std": float(lag_df["response_lag_days"].std()),
    "min": int(lag_df["response_lag_days"].min()),
    "p25": float(lag_df["response_lag_days"].quantile(0.25)),
    "p50": float(lag_df["response_lag_days"].quantile(0.50)),
    "p75": float(lag_df["response_lag_days"].quantile(0.75)),
    "max": int(lag_df["response_lag_days"].max()),
    "n_negative": int((lag_df["response_lag_days"] < 0).sum()),
    "n_null_date_received": int(lag_df["date_received_parsed"].null_count()),
    "n_null_date_sent": int(lag_df["date_sent_parsed"].null_count()),
}
date_received_range = {
    "min": str(lag_df["date_received_parsed"].min()),
    "max": str(lag_df["date_received_parsed"].max()),
}
date_sent_range = {
    "min": str(lag_df["date_sent_parsed"].min()),
    "max": str(lag_df["date_sent_parsed"].max()),
}
print(f"[OK] Real response_lag_days stats (live): {response_lag_stats}")
print(f"[OK] Real 'Date received' range (live): {date_received_range}")
print(f"[OK] Real 'Date sent to company' range (live): {date_sent_range}")

cluster_sizes = cfpb_lazy.group_by(ISSUE_CLUSTER_KEY, maintain_order=False).agg(pl.len().alias("n")).collect()
n_clusters = cluster_sizes.height
recurring = cluster_sizes.filter(pl.col("n") > 1)
n_recurring_clusters = recurring.height
max_cluster_size = int(cluster_sizes["n"].max())
rows_in_recurring_clusters = int(recurring["n"].sum())
issue_cluster_stats = {
    "cluster_key": ISSUE_CLUSTER_KEY,
    "n_clusters": int(n_clusters),
    "n_recurring_clusters": int(n_recurring_clusters),
    "pct_clusters_recurring": round(n_recurring_clusters / n_clusters, 4) if n_clusters else None,
    "max_cluster_size": max_cluster_size,
    "rows_in_recurring_clusters": rows_in_recurring_clusters,
    "pct_rows_in_recurring_clusters": (
        round(rows_in_recurring_clusters / total_rows, 4) if total_rows else None
    ),
}
print(f"[OK] Real issue-cluster stats (live): {issue_cluster_stats}")

null_counts_key_columns = {}
for _col in ["Company", "Product", "Sub-product", "Issue", "Sub-issue", "State", "ZIP code"]:
    null_counts_key_columns[_col] = int(
        cfpb_lazy.filter(pl.col(_col).is_null()).select(pl.len()).collect().item()
    )
print(f"[OK] Real null counts on journey-relevant columns (live): {null_counts_key_columns}")

# ============================================================
# SECTION 7: Assemble the Gate 1 policy (journey definition, scope boundaries, assumptions)
# ============================================================
policy = {
    "bp_id": "bp4",
    "bp_name": "bp4_customer_journey_analytics",
    "gate": 1,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "journey_definition": {
        "no_customer_identifier_in_scope": True,
        "naming_commitment": "BP4 is event/issue journey analytics, never customer/longitudinal "
        "journey analytics - stated in every BP4 deliverable from Gate 1 forward, per Master Plan "
        "Section 5.1/7's own explicit instruction for BP4.",
        "unit_1_complaint_event_journey": {
            "description": "Row-level, keyed by the real 'Complaint ID' (live-verified 1:1 with "
            "rows). A real two-step event sequence: 'Date received' -> 'Date sent to company', "
            "with a real computed 'response_lag_days' duration and a real terminal state "
            "('Company response to consumer', 'Timely response?').",
            "key_column": "Complaint ID",
            "sequence_columns": ["Date received", "Date sent to company"],
            "derived_metric": "response_lag_days = Date sent to company - Date received",
        },
        "unit_2_issue_cluster_journey": {
            "description": "Aggregate-level, keyed by the real, disclosed "
            "(Company, Product, Sub-product, Issue, Sub-issue) combination, ordered by "
            "'Date received' into a real monthly complaint-volume time series per cluster - the "
            "Master Plan's 'repeat-contact/bottleneck' signal, explicitly redefined as a "
            "recurring-issue-cluster signal, never a customer's own repeat-contact count.",
            "key_columns": ISSUE_CLUSTER_KEY,
            "secondary_overlay": "Gold-layer 'common_taxonomy_bucket' (derived BANKING77 "
            "integration per the Master Plan's BP table) - applied only to the real, honestly-"
            "scoped in-coverage subset (see banking77_derived_coverage below), never treated as "
            "covering the full real dataset.",
        },
        "banking77_derived_coverage": {
            "mechanism": "Reuses the Gold-layer 'common_taxonomy_bucket' column BP1/BP2's own "
            "Gate 2 work already built (HYPER) - never recomputed here, and BP1's text classifier "
            "is never run against CFPB rows, since this real extract carries no narrative-text "
            "column at all (live-verified in Section 4).",
            "n_in_scope": int(n_in_scope_banking77),
            "n_total": int(gold_row_count),
            "pct_in_scope": pct_in_scope_banking77,
            "bucket_distribution": common_taxonomy_bucket_counts.to_dicts(),
        },
        "explicitly_out_of_scope": [
            "Any per-customer or per-consumer journey, cohort, retention, or survival-style "
            "analysis - none is buildable without inventing an identity this real extract does "
            "not carry.",
            "Any claim that a real issue-cluster's recurrence count represents a single person's "
            "repeat contact.",
        ],
    },
    "scope_boundaries": [
        "'Complaint ID' identifies one real complaint (one event), never a person - must never be "
        "described, here or downstream, as a customer identifier.",
        "'Tags' is barred from every BP4 journey-grouping key - ECOA/Reg B is not a BP4 compliance "
        "touchpoint per Master Plan Section 9 (mapped to BP1, BP2, BP3, BP7 only), but this is kept "
        "as a conservative scope decision anyway, HYPER-consistent with BP1-BP3's own precedent of "
        "barring 'Tags' from any grouping/feature key.",
        "'ZIP code' is barred from every BP4 journey-grouping key - a fine-grained quasi-identifier "
        "(BP2/BP3's own BARRED_COLUMNS precedent); 'State' (coarser, already used as a plain "
        "categorical field by BP1-BP3) is the only geographic dimension BP4 may use.",
        "No BP4 gate may construct, infer, or approximate a persistent customer/consumer "
        "identifier from any combination of real fields (ZIP code + Company + Product + Issue + "
        "Date received or otherwise) - the issue-cluster journey (unit 2 above) is the only "
        "sanctioned aggregate-level grouping.",
        "No BP4 gate may run BP1's text-intent classifier against CFPB rows - this real extract "
        "carries no narrative-text column (live-verified in Section 4); the Gold-layer "
        "'common_taxonomy_bucket' overlay (already computed) is the only sanctioned BANKING77-"
        "derived signal, used only for its real in-scope subset.",
    ],
    "assumptions": [
        "This real CFPB extract carries no customer/consumer identifier of any kind "
        "(live-verified in Section 4, consistent with every prior BP1-BP3 Gate 1 finding on the "
        "same schema) - BP4's scope is built around this fact from Gate 1 forward, not worked "
        "around later.",
        "'Complaint ID' is a real, unique event identifier (live-verified 1:1 with rows in "
        "Section 4) - safe to use as the complaint-event journey's key, never as a person key.",
        "The (Company, Product, Sub-product, Issue, Sub-issue) issue-cluster key is real-data-"
        "supported: 37,160 real distinct clusters, 41.85% of which recur, covering 97.94% of all "
        "real rows (Section 6) - a well-supported aggregate signal, not a sparse or forced one.",
        "The Gold-layer 'common_taxonomy_bucket' column real-covers only 6.55% of rows "
        "(Section 5) - used as an optional secondary overlay only, never as BP4's primary "
        "journey-grouping dimension.",
        "'response_lag_days' (Date sent to company - Date received) is safe for BP4 to use "
        "directly, unlike BP3's Gate 1 barring of the same two Date fields: BP3 barred them as a "
        "predictive-leakage risk against a supervised target BP4 does not have. BP4 has no "
        "supervised target for this duration to leak into - it is reported descriptively, not "
        "used to predict 'Company response to consumer' or 'Timely response?'.",
        "src/utils/bp1_config_sync.py is reused unmodified for BP4's own config file - already "
        "fully generic, parameterized by config_path, no BP1-specific logic (already reused "
        "unmodified by BP2 and BP3 too).",
    ],
    "compliance_touchpoint": {
        "requirement": "CFPB supervisory & complaint-handling standards (Master Plan Section 9; "
        "governs all 8 BPs, primarily BP1-BP4)",
        "statement": "BP4's primary journey-grouping key (Company, Product, Sub-product, Issue, "
        "Sub-issue) is CFPB's own real product/issue/sub-issue schema, used as-is, with no "
        "fabricated outcomes anywhere. Two real dataset limitations are stated plainly rather than "
        "left implicit: (1) no customer/consumer identifier exists in this extract, so every "
        "'repeat-contact' or 'journey' claim in BP4's deliverables is an issue-cluster-level "
        "signal, never a per-person claim; (2) the Gold-layer BANKING77-derived taxonomy overlay "
        "real-covers only 6.55% of rows, so it is never presented as a population-level view of "
        "customer intent across the full real dataset.",
    },
    "live_checks": {
        "cfpb_row_count": total_rows,
        "cfpb_columns": cfpb_columns,
        "cfpb_columns_match_manifest": cfpb_columns == EXPECTED_CFPB_COLUMNS,
        "suspected_customer_id_columns": suspected_customer_id_columns,
        "narrative_text_columns": narrative_text_columns,
        "complaint_id_n_unique": complaint_id_n_unique,
        "complaint_id_is_unique_event_id": complaint_id_is_unique_event_id,
        "gold_taxonomy_parquet_row_count": gold_row_count,
        "gold_taxonomy_row_count_matches_raw": gold_row_count_matches_raw,
        "gold_taxonomy_columns": gold_columns,
        "response_lag_days_stats": response_lag_stats,
        "date_received_range": date_received_range,
        "date_sent_to_company_range": date_sent_range,
        "issue_cluster_stats": issue_cluster_stats,
        "null_counts_journey_columns": null_counts_key_columns,
    },
}

# ============================================================
# SECTION 8: Write outputs (idempotent overwrite-in-place)
# ============================================================
policy_json_path = ARTIFACTS_DIR / "policy.json"
with open(policy_json_path, "w", encoding="utf-8") as f:
    json.dump(policy, f, indent=2, default=str)
print(f"\n[SAVED] {policy_json_path.relative_to(PROJECT_ROOT)}")

bp4_config_path = CONFIGS_DIR / "bp4_customer_journey_analytics.yaml"

# BP4 reuses BP1's marker-based config-sync helpers as-is (src/utils/bp1_config_sync.py) - already
# generic, already reused unmodified by BP2 and BP3. Gate 1 here owns only the front-matter section
# below; every later gate's block (once written) is preserved verbatim regardless of position or
# order. See LESSONS_LEARNED_APPLIED.md #20 for the real incident this pattern was built to prevent.
from utils.bp1_config_sync import read_existing_gate_block_markers, write_front_matter  # noqa: E402

_existing_gate_markers = read_existing_gate_block_markers(bp4_config_path)
_status_suffix = ""
for _gate_num, _gate_label in ((2, "Gate 2"), (3, "Gate 3"), (4, "Gate 4"), (5, "Gate 5")):
    if any(_gate_label in _m for _m in _existing_gate_markers):
        _status_suffix += f"_gate{_gate_num}_confirmed"

bp4_config_text = f"""# Per-BP config - filled in at Gate 1 (Business Understanding & Policy)
# Gate 1 owns bp_id through random_state below via write_front_matter() (src/utils/bp1_config_sync.py,
# reused as-is from BP1/BP2/BP3 - fully generic, parameterized by config_path); Gates 2-5 each own
# exactly one marker-delimited block appended after it via write_gate_block() - do not hand-edit
# either section, re-run the owning notebook instead.
bp_id: "bp4"
bp_name: "bp4_customer_journey_analytics"
status: "gate1_confirmed{_status_suffix}"   # not_started|gate1|gate2|gate3|gate4|gate5|gate6_complete
journey_definition:
  no_customer_identifier_in_scope: true
  naming_commitment: "Event/issue journey analytics only - never customer/longitudinal journey
    analytics (Master Plan Section 5.1/7's own explicit instruction for BP4)."
  unit_1_complaint_event_journey: "Row-level, keyed by real 'Complaint ID'; 'Date received' ->
    'Date sent to company' with a real computed response_lag_days duration."
  unit_2_issue_cluster_journey: "Aggregate-level, keyed by real (Company, Product, Sub-product,
    Issue, Sub-issue); ordered by 'Date received' into a real monthly volume time series per
    cluster. Optionally overlaid with the Gold-layer 'common_taxonomy_bucket' column for its real
    6.55%-of-rows in-scope subset only."
scope_boundaries:
  - "'Complaint ID' identifies one real complaint (one event), never a person."
  - "'Tags' barred from every BP4 journey-grouping key (conservative, HYPER-consistent with
     BP1-BP3's own precedent, though ECOA/Reg B does not map to BP4 per Master Plan Section 9)."
  - "'ZIP code' barred as a fine-grained quasi-identifier; 'State' is the only sanctioned
     geographic dimension."
  - "No BP4 gate may construct, infer, or approximate a persistent customer/consumer identifier
     from any combination of real fields."
  - "No BP4 gate may run BP1's text-intent classifier against CFPB rows - this extract carries no
     narrative-text column; only the already-built Gold-layer 'common_taxonomy_bucket' overlay is
     sanctioned."
assumptions:
  - "This real CFPB extract carries no customer/consumer identifier of any kind (live-verified,
     consistent with every prior BP1-BP3 Gate 1 finding on the same schema)."
  - "'Complaint ID' is a real, unique event identifier (live-verified 1:1 with rows)."
  - "The (Company, Product, Sub-product, Issue, Sub-issue) issue-cluster key is real-data-
     supported: 37,160 real distinct clusters, 41.85% recurring, covering 97.94% of all real rows."
  - "The Gold-layer 'common_taxonomy_bucket' column real-covers only 6.55% of rows - used as an
     optional secondary overlay only."
  - "'response_lag_days' is safe for BP4 to use directly (unlike BP3's leakage-driven bar on the
     same two Date fields) - BP4 has no supervised target for this duration to leak into."
  - "src/utils/bp1_config_sync.py reused unmodified for BP4's own config file."
random_state: 42
"""
write_front_matter(bp4_config_path, bp4_config_text)
print(f"[SAVED] {bp4_config_path.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 9: Structural integrity checks - raise AssertionError, never silently pass
# ============================================================
checks = {
    "cfpb_schema_matches_manifest": cfpb_columns == EXPECTED_CFPB_COLUMNS,
    "no_customer_identifier_column_in_cfpb_schema": len(suspected_customer_id_columns) == 0,
    "no_narrative_text_column_in_cfpb_schema": len(narrative_text_columns) == 0,
    "complaint_id_is_unique_event_identifier": complaint_id_is_unique_event_id,
    "gold_taxonomy_parquet_exists": GOLD_TAXONOMY_PATH.exists(),
    "gold_taxonomy_row_count_matches_raw": gold_row_count_matches_raw,
    "common_taxonomy_bucket_enumerated": common_taxonomy_bucket_counts.height > 0,
    "response_lag_days_no_negative_values": response_lag_stats["n_negative"] == 0,
    "response_lag_days_no_null_dates": (
        response_lag_stats["n_null_date_received"] == 0 and response_lag_stats["n_null_date_sent"] == 0
    ),
    "at_least_one_recurring_issue_cluster": n_recurring_clusters > 0,
    "issue_cluster_row_accounting_within_total": rows_in_recurring_clusters <= total_rows,
    "policy_json_written": policy_json_path.exists(),
    "bp4_config_yaml_written": bp4_config_path.exists(),
}

print("\n=== INTEGRITY CHECKS ===")
for name, passed in checks.items():
    status = "[PASS]" if passed else "[FAIL]"
    print(f"{status} {name}")
    assert passed, f"[CHECK FAILED] {name}"

print(
    "\n[ALL CHECKS PASSED] BP4 Gate 1 complete - no customer identifier confirmed absent (live), "
    "'Complaint ID' confirmed a real unique event id, the real (Company, Product, Sub-product, "
    "Issue, Sub-issue) issue-cluster journey key confirmed well-supported, real response_lag_days "
    "computed with zero negative/null anomalies, and the Gold-layer BANKING77-derived taxonomy "
    "overlay's real 6.55% coverage stated honestly. Proceed to BP4 Gate 2 (Data Verification & "
    "Feature/Taxonomy Engineering) next."
)
